In [1]:
import faiss
import spacy
import re
import contractions
from textblob import TextBlob
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

d:\Study\pyspider\GenAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Load the document(.txt)

In [2]:
data = open('data.txt', 'r').read()

### 2. Text Normalization

#### Converting text into lowercase

In [3]:
data = data.lower()

### removing numbers like 1., 2. 

In [4]:
data = re.sub('\d+\.',"", data)
data = re.sub(r"\b\d+(?:\.\d+)*\.?\b", "", data)
data

'====================================================================\nartificial intelligence (ai), machine learning (ml) and\ndeep learning (dl) - a complete guide\n====================================================================\n\ntable of contents\n------------------\n introduction\n what is artificial intelligence (ai)?\n history and evolution of ai\n types of artificial intelligence\n what is machine learning (ml)?\n how machine learning works\n types of machine learning\n    supervised learning\n    unsupervised learning\n    semi-supervised learning\n    reinforcement learning\n common machine learning algorithms\n what is deep learning (dl)?\n how deep learning works\n neural network architectures\n difference between ai, ml and dl\n applications of ai, ml and dl\n popular tools and frameworks\n challenges and limitations\n ethics and responsible ai\n future trends\n conclusion\n glossary of terms\n\n====================================================================\n i

#### expanding the words

In [5]:
data = contractions.fix(data)

#### removing punctuations and spl characters

In [6]:
data = re.sub('[^0-9a-zA-Z\s]', "", data).strip()
data

'artificial intelligence ai machine learning ml and\ndeep learning dl  a complete guide\n\n\ntable of contents\n\n introduction\n what is artificial intelligence ai\n history and evolution of ai\n types of artificial intelligence\n what is machine learning ml\n how machine learning works\n types of machine learning\n    supervised learning\n    unsupervised learning\n    semisupervised learning\n    reinforcement learning\n common machine learning algorithms\n what is deep learning dl\n how deep learning works\n neural network architectures\n difference between ai ml and dl\n applications of ai ml and dl\n popular tools and frameworks\n challenges and limitations\n ethics and responsible ai\n future trends\n conclusion\n glossary of terms\n\n\n introduction\n\n\nartificial intelligence machine learning and deep learning are three\nof the most talkedabout technologies of the twentyfirst century\nthey are reshaping industries changing how businesses operate and\ninfluencing everyday life

### removing extra spaces

In [7]:
data = re.sub("\s+", " ", data).strip()
data

'artificial intelligence ai machine learning ml and deep learning dl a complete guide table of contents introduction what is artificial intelligence ai history and evolution of ai types of artificial intelligence what is machine learning ml how machine learning works types of machine learning supervised learning unsupervised learning semisupervised learning reinforcement learning common machine learning algorithms what is deep learning dl how deep learning works neural network architectures difference between ai ml and dl applications of ai ml and dl popular tools and frameworks challenges and limitations ethics and responsible ai future trends conclusion glossary of terms introduction artificial intelligence machine learning and deep learning are three of the most talkedabout technologies of the twentyfirst century they are reshaping industries changing how businesses operate and influencing everyday life in ways most people do not even notice from voice assistants like siri and alexa

In [8]:
corrected_words = TextBlob(data).correct()

In [9]:
data = str(corrected_words)
data

'artificial intelligence ai machine learning my and deep learning do a complete guide table of contents introduction what is artificial intelligence ai history and evolution of ai types of artificial intelligence what is machine learning my how machine learning works types of machine learning supervised learning supervised learning semisupervised learning reinforcement learning common machine learning algorithms what is deep learning do how deep learning works neutral network architecture difference between ai my and do applications of ai my and do popular tools and framework challenges and limitations ethics and responsible ai future tends conclusion glossy of terms introduction artificial intelligence machine learning and deep learning are three of the most talkedabout technologies of the twentyfirst century they are escaping industries changing how business operate and influencing everyday life in ways most people do not even notice from voice assistants like sir and area to recomme

#### lematization, removing stopwords

In [10]:
nlp = spacy.load('en_core_web_sm')

In [11]:
tokens = nlp(data)
lemmatize_tokens = [token.lemma_ for token in tokens if not token.is_stop]
data = " ".join(lemmatize_tokens).strip()
data

'artificial intelligence ai machine learn deep learning complete guide table content introduction artificial intelligence ai history evolution ai type artificial intelligence machine learn machine learn work type machine learning supervise learning supervise learning semisupervise learn reinforcement learn common machine learning algorithm deep learning deep learning work neutral network architecture difference ai application ai popular tool framework challenge limitation ethic responsible ai future tend conclusion glossy term introduction artificial intelligence machine learning deep learning talkedabout technology twentyfirst century escape industry change business operate influence everyday life way people notice voice assistant like sir area recommendation system netflix amazon selfdrive car medical diagnosis tool technology term interchangeable casual conversation thing artificial intelligence broad concept machine learning sunset ai deep learning sunset machine learning understan

#### chunking

In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, 
    chunk_overlap=40, 
    separators=["\n\n","\n",".","!"," ",""])
data = text_splitter.split_text(data)

#### chunk Embeddings

In [13]:
embeddings_model = SentenceTransformer(
    model_name_or_path = 'sentence-transformers/all-miniLM-L6-V2'
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3560.47it/s]


In [14]:
embeddings = embeddings_model.encode(data).astype('float32')

In [15]:
embeddings

array([[-0.03389303, -0.07876014,  0.06862532, ...,  0.05817775,
        -0.00551835, -0.016296  ],
       [-0.10913681, -0.01691784,  0.06696158, ...,  0.03125955,
         0.05486624,  0.05000151],
       [-0.06983884, -0.02376575,  0.01137985, ..., -0.05219284,
         0.00449924, -0.00790474],
       ...,
       [-0.05453707, -0.02638213, -0.06061979, ...,  0.02220731,
         0.01549557, -0.03317378],
       [-0.05741737, -0.01729369, -0.04833126, ...,  0.0474743 ,
         0.03010154, -0.019301  ],
       [ 0.03132785, -0.04969292, -0.0257167 , ...,  0.10944637,
         0.04736606,  0.02910013]], shape=(115, 384), dtype=float32)

In [16]:
embeddings.shape

(115, 384)

In [17]:
dimension = embeddings.shape[1]
dimension

384

In [18]:
faiss.normalize_L2(embeddings)

In [19]:
index_faiss_db = faiss.IndexFlatIP(dimension) # dot product 

In [20]:
index_faiss_db.add(embeddings)

In [21]:
def rag_query(query, k=2):
    # faiss.normalize_L2(query)
    query_embeddings = embeddings_model.encode(query).astype('float32')
    query_embeddings = query_embeddings.reshape(1, -1)
    faiss.normalize_L2(query_embeddings)
    distance, index =index_faiss_db.search(query_embeddings, k=k)# search_documents
    # for i in index[0]:
    #     print(data[i])
    # print(index)
    r_chunks = [data[i] for i in index[0]]
    r_string = " ".join(r_chunks)
    
    prompt = f'''
              You're an helpful assistant
              Assigned Task for you : structure my output => {r_string}
              note :
               1) don't add extra contents just structure mentioned output
               2) if there is mistake in output correct or else keep the original output
               with structures result.

    '''
    import os
    import requests

    API_URL = "https://router.huggingface.co/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
    }

    def query(payload):
        response = requests.post(API_URL, headers=headers, json=payload)
        return response.json()

    response = query({
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "model": "deepseek-ai/DeepSeek-V4-Pro:novita"
    })
    return response
user_prompt = "Explain machine learning ?"
user_prompt = re.sub('^0-9a-zA-Z', "", user_prompt)

response = rag_query(user_prompt)
print(response)

{'id': '70f941c08653afef12797942c8d950c1', 'object': 'chat.completion', 'created': 1785814383, 'model': 'deepseek/deepseek-v4-pro', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '**Decision Term: Machine Learning**\n\n**Join Arthur Samuel\'s Definition**\n- Machine learning is the field of study that gives computers the ability to learn without being explicitly programmed.\n- Traditional programming involves writing explicit rules to tell the computer exactly what to do.\n\n**Intelligence of Machine Learning**\n- How machine learning works.\n\n**Types of Machine Learning**\n- Supervised learning\n- Unsupervised learning (implied by "semisupervise" correction to "semi-supervised learning")\n- Semi-supervised learning\n- Reinforcement learning\n\n**Common Machine Learning Algorithms**\n- Deep learning (deep neural networks)', 'reasoning_content': 'We are asked to structure the given output. The input is a string: "structure my output => decision term machine learni

In [22]:
response['choices'][0]['message']['content']

'**Decision Term: Machine Learning**\n\n**Join Arthur Samuel\'s Definition**\n- Machine learning is the field of study that gives computers the ability to learn without being explicitly programmed.\n- Traditional programming involves writing explicit rules to tell the computer exactly what to do.\n\n**Intelligence of Machine Learning**\n- How machine learning works.\n\n**Types of Machine Learning**\n- Supervised learning\n- Unsupervised learning (implied by "semisupervise" correction to "semi-supervised learning")\n- Semi-supervised learning\n- Reinforcement learning\n\n**Common Machine Learning Algorithms**\n- Deep learning (deep neural networks)'

In [23]:
'**Machine Learning Overview**\n\n- **Definition (Arthur Samuel):**  \n  Field of study that gives computers the ability to learn without being explicitly programmed.\n\n- **Traditional Programming vs. Machine Learning:**  \n  - Traditional programming: writes explicit rules telling the computer exactly what to do.  \n  - Machine learning: computers learn from data.\n\n- **Types of Machine Learning:**  \n  - Supervised Learning  \n  - Semi-Supervised Learning  \n  - Reinforcement Learning\n\n- **Common Machine Learning Algorithms:**  \n  - Deep Learning (deep neural networks)'

'**Machine Learning Overview**\n\n- **Definition (Arthur Samuel):**  \n  Field of study that gives computers the ability to learn without being explicitly programmed.\n\n- **Traditional Programming vs. Machine Learning:**  \n  - Traditional programming: writes explicit rules telling the computer exactly what to do.  \n  - Machine learning: computers learn from data.\n\n- **Types of Machine Learning:**  \n  - Supervised Learning  \n  - Semi-Supervised Learning  \n  - Reinforcement Learning\n\n- **Common Machine Learning Algorithms:**  \n  - Deep Learning (deep neural networks)'